# Lesson 12 - Chat History Reduction wit Agent Scratchpad

Dis notebook dey show how to manage context for long conversations using Microsoft Agent Framework. As conversations dee grow, token count go increase — and e fit pass di model's context window. We go solve dis wit **context summarization pattern** and **agent scratchpad** wey get persistent memory.

## Wetin You Go Learn:
1. **Why Context Management Matter**: Understand token limits and context windows
2. **Context-Aware Agents**: Build agents wey dey manage their own conversation context
3. **Context Summarization Pattern**: Use tools to condense conversation history
4. **Agent Scratchpad**: Persistent memory wey dey survive context reduction

## Prerequisites:
- Azure OpenAI setup with environment variables configured
- Understand basic agent concepts from previous lessons


## Setup


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity --quiet

In [ ]:
import os
import asyncio
from datetime import datetime
from pathlib import Path

from dotenv import load_dotenv

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Load environment variables and create Azure AI Foundry provider
load_dotenv()

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("✅ Azure AI Foundry provider configured")


## Why Context Management Matters

Every LLM get one limited **context window** — na di maximum number of tokens e fit process for one single request. As conversation wey dey multi-turn dey go:

- **Token count dey grow linearly** with each user message plus assistant reply.
- **Prompt tokens dey dominate cost** because dem go dey resend all di history every turn.
- At last di conversation go **pass di context window** and di model go either cut am short or give error.

### Strategies for Managing Context

| Strategy | How E Dey Work | Trade-off |
|---|---|---|
| **Truncation** | Drop di oldest messages | Go lose early context |
| **Summarization** | Condense older messages into summary | Some detail go lost, but key points go still dey |
| **Scratchpad / External Memory** | Store key facts outside di conversation | Need tool calls, but e fit survive any kind reduction |

For dis notebook, we combine **summarization** with **scratchpad tool** so di agent fit maintain continuity even if conversation history dey condensed.


## Dey Create One Context-Aware Agent


In [ ]:
agent = await provider.create_agent(
    name="ContextAwareAgent",
    instructions="""You are a helpful travel planning assistant with excellent memory management.
When conversations get long:
1. Summarize previous context into key points
2. Track user preferences mentioned earlier
3. Reference previous decisions without repeating full details
Always maintain continuity while being concise.""",
)

print("🤖 Context-aware travel planning agent created")

## Simulating a Long Conversation

Make we waka through one kain long talk wey get many turns to see as how context dey add up. The agent suppose keep important tins (preferences, budget, travel dates) for all turns and show say e get continuity.


In [ ]:
session = agent.create_session()

# Turn 1 - Initial preferences
response = await agent.run("I'm planning a trip to Japan. I love sushi, temples, and photography.", session=session)
print(f"Turn 1: {response}\n")

# Turn 2 - More details
response = await agent.run("My budget is $3000 and I'll be traveling solo for 10 days in April.", session=session)
print(f"Turn 2: {response}\n")

# Turn 3 - Test context retention
response = await agent.run("Based on everything I've told you so far, what's the one thing you'd recommend I not miss?", session=session)
print(f"Turn 3: {response}\n")

Notice how di agent dey hold context from before turns — e sabi about Japan, sushi, temples, photography, di $3000 budget, solo travel, and di April timeframe. For short conversation, dis one dey work well, but as di conversation long, di full history go begin cost plenty to re-send.

Make we continue di conversation wit more turns to see how context dey accumulate:


In [ ]:
# Turn 4 - Expand the conversation
response = await agent.run("What about accommodation? I prefer traditional Japanese inns.", session=session)
print(f"Turn 4: {response}\n")

# Turn 5 - Change of plans
response = await agent.run("Actually, I've changed my mind about the dates. I'll go in October instead for the autumn colors.", session=session)
print(f"Turn 5: {response}\n")

# Turn 6 - Test retention after change
response = await agent.run("Summarize my complete travel plan so far — destination, budget, duration, interests, accommodation, and timing.", session=session)
print(f"Turn 6: {response}\n")

## Context Summarization Pattern

As di konversations dey grow, we fit use **summarization tool** to make di accumulated context short and tight. Di agent go call dis tool to record di main preferences so dat even if old messages drop, di important info still dey safe.

Dis pattern na di building block for better history reduction:
1. Di agent go find main facts from di konversation
2. E go call di summarization tool to save dem
3. Old messages fit comot safely because di summary dey hold wetin matter

Below we define one `summarize_preferences` tool wey di agent fit call to record one tight summary of wetin e don learn.


In [ ]:
@tool(approval_mode="never_require")
def summarize_preferences(conversation_notes: str) -> str:
    """Summarize accumulated user preferences into a compact format."""
    return f"[SUMMARY] User preferences recorded: {conversation_notes}"


# Create an enhanced agent with the summarization tool
summarizing_agent = await provider.create_agent(
    name="SummarizingTravelAgent",
    instructions="""You are a helpful travel planning assistant that actively manages conversation context.

CONTEXT MANAGEMENT RULES:
1. After gathering several user preferences, call summarize_preferences() to record a compact summary
2. When the user asks you to recall details, reference your recorded summaries
3. Keep responses concise — avoid restating the entire history

PLANNING PROCESS:
1. Gather user preferences (destination, budget, dates, interests)
2. Summarize preferences using the tool
3. Create recommendations based on the summary
4. Update the summary when preferences change""",
    tools=[summarize_preferences],
)

print("🤖 Summarizing travel agent created with context tools")

In [ ]:
# Demonstrate the summarization pattern
summary_session = summarizing_agent.create_session()

# Provide a batch of preferences
response = await summarizing_agent.run(
    "I want to visit Greece. I love seafood, history, and island hopping. "
    "Budget is $4000 for two weeks. Traveling with my partner in June. "
    "Please record these preferences using your summarization tool.",
    session=summary_session,
)
print(f"Agent: {response}\n")

# Ask the agent to use the recorded context
response = await summarizing_agent.run(
    "Now, based on what you've recorded, suggest the top 3 islands we should visit.",
    session=summary_session,
)
print(f"Agent: {response}\n")

## Summary

For dis lesson, you learn how to manage context for long-running agent conversations using Microsoft Agent Framework:

### Key Concepts
- **Context windows na limited** — every token wey dey inside conversation history dey cost money and dey count towards the limit.
- **Summarization tools** dey allow agent to condense accumulated context into small summaries, wey go reduce token usage but still keep di important information.
- **Agent scratchpads** dey give persistent external memory wey fit survive any conversation reduction.

### Wetin You Build
- A **context-aware agent** wey go maintain continuity through multi-turn conversations
- A **summarization tool** (`summarize_preferences`) wey dey record key user details for small format
- A **multi-turn conversation** wey show how context dey retained and how e dey handle change

### Real-World Applications
- **Customer Service Bots**: Dey remember preferences across long support sessions
- **Personal Assistants**: Dey track ongoing projects without you needing to explain context again
- **Educational Tutors**: Dey maintain student progress through many interactions

### Next Steps
- Implement full scratchpad tool wey get file-based persistence
- Add automatic history truncation after summarization
- Combine with vector databases for semantic memory search
- Build agents wey fit resume conversations days later with full context


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Disclaimer**:
Dis document na AI translation service [Co-op Translator](https://github.com/Azure/co-op-translator) wey translate am. Even tho we dey try make am correct, abeg sabi say automated translation fit get mistakes or no correct well. Di original document wey e dey for im own language na di correct one wey you suppose trust. If na serious tins, e better make person wey sabi do translation translate am. We no go carry any wahala if person no understand well or if dem misunderstand because of dis translation.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
